In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical
from dataclasses import dataclass

from balatro_gym.balatro_env_2 import BalatroEnv
from cs590_env.wrapper import BalatroPhaseWrapper
from cs590_env.schema import (
    WrapperAction, GamePhase, MAX_HAND_SIZE,
    SELECT_CARD_COUNT, get_wrapper_select_action,
)

%run CombatAgent.ipynb


@dataclass
class PPOConfig:
    lr: float             = 2.5e-4
    gamma: float          = 0.99
    gae_lambda: float     = 0.95
    clip_eps: float       = 0.2
    ppo_epochs: int       = 4
    num_minibatches: int  = 4
    rollout_steps: int    = 128
    max_iterations: int   = 1000
    c_value: float        = 0.5
    c_entropy: float      = 0.01
    max_grad_norm: float  = 0.5
    d_model: int          = 256
    nhead: int            = 8
    dim_ff: int           = 1024
    dropout: float        = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = PPOConfig()

In [ ]:
class CombatActionWrapper:
    """Bridges the factored action space (8 binary card selections + play/discard)
    with the sequential toggle-based BalatroPhaseWrapper.

    On reset, auto-advances through non-combat phases (blind select, shop)
    so the agent always sees a COMBAT observation.

    On step, translates a factored action into sequential env steps:
      1. Toggle cards to match the desired selection
      2. Execute play (action 0) or discard (action 1)
    """

    def __init__(self, env: BalatroPhaseWrapper):
        self.env = env
        self._last_obs = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        obs = self._advance_to_combat(obs)
        self._last_obs = obs
        return obs, info

    def _advance_to_combat(self, obs):
        """Step through non-combat phases with default actions until
        we reach COMBAT."""
        while True:
            phase = GamePhase(int(obs['phase']))
            if phase == GamePhase.COMBAT:
                return obs

            if phase == GamePhase.TRANSITION:
                mask = obs['action_mask']
                for i in range(3):
                    if mask[WrapperAction.SELECT_BLIND_BASE + i]:
                        obs, _, done, _, _ = self.env.step(
                            int(WrapperAction.SELECT_BLIND_BASE + i))
                        break
            elif phase == GamePhase.SHOP:
                obs, _, done, _, _ = self.env.step(int(WrapperAction.SHOP_END))
            else:
                break

            if done:
                obs, _ = self.env.reset()
        return obs

    def step(self, card_selections: np.ndarray, execution: int):
        """Execute a factored combat action.

        Args:
            card_selections: (MAX_HAND_SIZE,) binary array — 1 = select, 0 = don't select
            execution:       0 = play, 1 = discard

        Returns:
            obs, reward, done, info
        """
        n_selected = int(card_selections.sum())
        if n_selected < 1 or n_selected > 5:
            return self._last_obs, -1.0, False, {
                'error': f'Invalid selection count: {n_selected}'}

        current_sel = self._last_obs['hand_is_selected']
        to_toggle = np.where(card_selections != current_sel)[0]

        total_reward = 0.0
        obs = self._last_obs
        done = False
        info = {}

        for idx in to_toggle:
            if idx >= SELECT_CARD_COUNT:
                continue
            action = get_wrapper_select_action(int(idx))
            obs, r, done, _, info = self.env.step(action)
            total_reward += r
            if done:
                self._last_obs = obs
                return obs, total_reward, True, info

        exec_action = int(WrapperAction.PLAY_HAND) if execution == 0 \
                      else int(WrapperAction.DISCARD)
        obs, r, done, _, info = self.env.step(exec_action)
        total_reward += r

        if not done:
            phase = GamePhase(int(obs['phase']))
            if phase != GamePhase.COMBAT:
                done = True
                info['combat_ended'] = True

        self._last_obs = obs
        return obs, total_reward, done, info

In [ ]:
def obs_to_tensors(obs: dict, dev: torch.device) -> dict:
    """Convert a single numpy observation dict to a batched torch tensor dict.

    Scalars become shape (1,); arrays become (1, *shape).
    """
    out = {}
    for k, v in obs.items():
        t = torch.as_tensor(np.asarray(v), device=dev)
        out[k] = t.unsqueeze(0) if t.dim() == 0 else t.unsqueeze(0)
    return out


def batch_obs(obs_list: list, dev: torch.device) -> dict:
    """Stack a list of single-obs tensor dicts into a batched dict.

    Each value goes from list of (1, ...) to (B, ...).
    """
    keys = obs_list[0].keys()
    return {k: torch.cat([o[k] for o in obs_list], dim=0).to(dev) for k in keys}


def get_card_mask(obs_t: dict) -> torch.Tensor:
    """(B, MAX_HAND_SIZE) bool — True for real cards, False for padding."""
    return obs_t['hand_card_ids'] >= 0


def mask_logits(sel_logits: torch.Tensor, card_mask: torch.Tensor) -> torch.Tensor:
    """Zero-out the 'select' logit for empty card slots.

    Args:
        sel_logits: (B, MAX_HAND_SIZE, 2) — raw logits from the model
        card_mask:  (B, MAX_HAND_SIZE) bool — True for real cards

    Returns:
        masked copy (does not modify in place, safe for autograd).
    """
    masked = sel_logits.clone()
    masked[:, :, 1] = masked[:, :, 1].masked_fill(~card_mask, -1e9)
    return masked

In [ ]:
class RolloutBuffer:
    """Collects trajectory data during rollout and serves batched
    minibatches for the PPO update.

    Stores observations as tensor dicts (already on device), plus
    flat tensors for actions, log_probs, values, rewards, and dones.
    """

    def __init__(self):
        self.clear()

    def clear(self):
        self.obs_list: list[dict] = []
        self.card_sels: list[torch.Tensor] = []
        self.executions: list[torch.Tensor] = []
        self.log_probs: list[torch.Tensor] = []
        self.values: list[torch.Tensor] = []
        self.rewards: list[float] = []
        self.dones: list[bool] = []

    def store(
        self,
        obs_t: dict,
        card_sel: torch.Tensor,
        execution: torch.Tensor,
        log_prob: torch.Tensor,
        value: torch.Tensor,
        reward: float,
        done: bool,
    ):
        self.obs_list.append(obs_t)
        self.card_sels.append(card_sel.detach())
        self.executions.append(execution.detach())
        self.log_probs.append(log_prob.detach())
        self.values.append(value.detach())
        self.rewards.append(reward)
        self.dones.append(done)

    def __len__(self):
        return len(self.rewards)

    def get(self, dev: torch.device):
        """Return all stored data as contiguous tensors."""
        batched_obs    = batch_obs(self.obs_list, dev)
        card_sels      = torch.stack(self.card_sels).to(dev)          # (T, MAX_HAND_SIZE)
        executions     = torch.stack(self.executions).to(dev)          # (T,)
        old_log_probs  = torch.stack(self.log_probs).to(dev)          # (T,)
        values         = torch.stack(self.values).squeeze(-1).to(dev)  # (T,)
        rewards        = torch.tensor(self.rewards, dtype=torch.float32, device=dev)
        dones          = torch.tensor(self.dones, dtype=torch.float32, device=dev)
        return batched_obs, card_sels, executions, old_log_probs, values, rewards, dones

In [ ]:
@torch.no_grad()
def compute_gae(
    rewards: torch.Tensor,     # (T,)
    values: torch.Tensor,      # (T,)
    next_value: torch.Tensor,  # scalar
    dones: torch.Tensor,       # (T,)
    gamma: float,
    gae_lambda: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Generalized Advantage Estimation.

    Returns:
        advantages: (T,)  normalized to zero mean, unit std
        returns:    (T,)  = advantages + values (before normalization)
    """
    T = len(rewards)
    advantages = torch.zeros_like(rewards)
    last_gae = 0.0

    for t in reversed(range(T)):
        next_val = next_value if t == T - 1 else values[t + 1]
        next_non_done = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_val * next_non_done - values[t]
        last_gae = delta + gamma * gae_lambda * next_non_done * last_gae
        advantages[t] = last_gae

    returns = advantages + values
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return advantages, returns

In [ ]:
def compute_log_prob_and_entropy(
    agent: CombatPPOAgent,
    obs_batch: dict,
    card_sels: torch.Tensor,
    executions: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Forward the agent and compute log-probs / entropy for stored actions.

    Returns:
        log_probs: (B,)
        entropy:   (B,)
        values:    (B,)
    """
    sel_logits, exec_logits, values = agent(obs_batch)
    card_mask = get_card_mask(obs_batch)
    sel_logits = mask_logits(sel_logits, card_mask)

    sel_dist  = Categorical(logits=sel_logits)        # independent (B, MAX_HAND_SIZE) binary dists
    exec_dist = Categorical(logits=exec_logits)       # (B,) over {play, discard}

    log_probs = (sel_dist.log_prob(card_sels).sum(dim=-1)
                 + exec_dist.log_prob(executions))    # (B,)
    entropy   = (sel_dist.entropy().sum(dim=-1)
                 + exec_dist.entropy())               # (B,)
    return log_probs, entropy, values.squeeze(-1)


def ppo_update(
    agent: CombatPPOAgent,
    optimizer: torch.optim.Optimizer,
    obs_batch: dict,
    card_sels: torch.Tensor,
    executions: torch.Tensor,
    old_log_probs: torch.Tensor,
    advantages: torch.Tensor,
    returns: torch.Tensor,
    cfg: PPOConfig,
) -> dict:
    """Run multiple PPO epochs over the collected rollout.

    Returns:
        dict of mean losses for logging.
    """
    T = len(advantages)
    batch_size = T // cfg.num_minibatches
    total_pg, total_vf, total_ent = 0.0, 0.0, 0.0
    num_updates = 0

    for _ in range(cfg.ppo_epochs):
        indices = torch.randperm(T, device=advantages.device)

        for start in range(0, T, batch_size):
            end = start + batch_size
            if end > T:
                break
            mb_idx = indices[start:end]

            mb_obs = {k: v[mb_idx] for k, v in obs_batch.items()}
            mb_card_sels    = card_sels[mb_idx]
            mb_executions   = executions[mb_idx]
            mb_old_lp       = old_log_probs[mb_idx]
            mb_advantages   = advantages[mb_idx]
            mb_returns      = returns[mb_idx]

            curr_lp, entropy, curr_values = compute_log_prob_and_entropy(
                agent, mb_obs, mb_card_sels, mb_executions)

            ratio = torch.exp(curr_lp - mb_old_lp)
            surr1 = ratio * mb_advantages
            surr2 = torch.clamp(ratio, 1 - cfg.clip_eps, 1 + cfg.clip_eps) * mb_advantages
            pg_loss    = -torch.min(surr1, surr2).mean()
            value_loss = nn.functional.mse_loss(curr_values, mb_returns)
            ent_bonus  = entropy.mean()

            loss = pg_loss + cfg.c_value * value_loss - cfg.c_entropy * ent_bonus

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), cfg.max_grad_norm)
            optimizer.step()

            total_pg  += pg_loss.item()
            total_vf  += value_loss.item()
            total_ent += ent_bonus.item()
            num_updates += 1

    return {
        'pg_loss':    total_pg  / max(num_updates, 1),
        'value_loss': total_vf  / max(num_updates, 1),
        'entropy':    total_ent / max(num_updates, 1),
    }

In [ ]:
# ── Initialise agent, env, optimizer ─────────────────────────────

agent = CombatPPOAgent(
    d_model=cfg.d_model, nhead=cfg.nhead,
    dim_ff=cfg.dim_ff, dropout=cfg.dropout,
).to(device)

optimizer = torch.optim.Adam(agent.parameters(), lr=cfg.lr, eps=1e-5)
buffer    = RolloutBuffer()

base_env    = BalatroEnv()
wrapped_env = BalatroPhaseWrapper(base_env)
env         = CombatActionWrapper(wrapped_env)


# ── Training loop ────────────────────────────────────────────────

obs, _ = env.reset()
ep_rewards = []
ep_reward  = 0.0
ep_lengths = []
ep_len     = 0

for iteration in range(1, cfg.max_iterations + 1):

    # ── Phase 1: Rollout collection ──────────────────────────────
    agent.eval()
    for _ in range(cfg.rollout_steps):
        obs_t = obs_to_tensors(obs, device)
        card_mask = get_card_mask(obs_t)

        with torch.no_grad():
            sel_logits, exec_logits, value = agent(obs_t)

        sel_logits = mask_logits(sel_logits, card_mask)

        sel_dist  = Categorical(logits=sel_logits)
        exec_dist = Categorical(logits=exec_logits)

        card_sel  = sel_dist.sample().squeeze(0)       # (MAX_HAND_SIZE,)
        execution = exec_dist.sample().squeeze(0)      # scalar

        log_prob = (sel_dist.log_prob(card_sel.unsqueeze(0)).sum(dim=-1)
                    + exec_dist.log_prob(execution.unsqueeze(0))).squeeze(0)

        next_obs, reward, done, info = env.step(
            card_sel.cpu().numpy(), execution.item())

        buffer.store(obs_t, card_sel, execution, log_prob,
                     value.squeeze(0), reward, done)

        ep_reward += reward
        ep_len += 1
        obs = next_obs

        if done:
            ep_rewards.append(ep_reward)
            ep_lengths.append(ep_len)
            ep_reward, ep_len = 0.0, 0
            obs, _ = env.reset()

    # ── Phase 2: GAE + PPO update ────────────────────────────────
    with torch.no_grad():
        next_obs_t = obs_to_tensors(obs, device)
        _, _, next_value = agent(next_obs_t)
        next_value = next_value.squeeze()

    (obs_batch, card_sels, executions,
     old_log_probs, values, rewards, dones) = buffer.get(device)

    advantages, returns = compute_gae(
        rewards, values, next_value, dones,
        cfg.gamma, cfg.gae_lambda)

    agent.train()
    losses = ppo_update(
        agent, optimizer,
        obs_batch, card_sels, executions, old_log_probs,
        advantages, returns, cfg)

    buffer.clear()

    # ── Logging ──────────────────────────────────────────────────
    if iteration % 10 == 0 or iteration == 1:
        mean_r   = np.mean(ep_rewards[-20:]) if ep_rewards else 0.0
        mean_len = np.mean(ep_lengths[-20:]) if ep_lengths else 0.0
        print(f"[iter {iteration:4d}]  "
              f"reward={mean_r:+.2f}  len={mean_len:.1f}  "
              f"pg={losses['pg_loss']:.4f}  "
              f"vf={losses['value_loss']:.4f}  "
              f"ent={losses['entropy']:.4f}")